# TSARA on the campaign archive — alignment on real data

**Phase 4, the companion to notebook 04.** Notebook 04 runs every alignment
operation on data manufactured inside it, so every answer can be checked against
the truth that produced it. This notebook runs the same operations on the real
campaign archive, where there is no answer key, so that you can see them work on
the records they were built for.

Without an answer key the checks are of three other kinds:

* **an independent reimplementation**, written from the definition in a loop and
  compared with TSARA's answer;
* **the numbers `docs/METHODS.md` §11 quotes**, re-measured here from the files.
  Every such number is collected as it is measured, and the last section lists
  each one beside the documented value and says whether they still agree;
* **physical plausibility**, in the figures.

**What it needs.** Set `TSARA_ARCHIVE` to the directory holding the archive's
`2024/` and `2026/` trees, then start Jupyter from that shell:

```bash
export TSARA_ARCHIVE=/path/to/Data
```

It reads only `2024/` (the NOAA mobile-lab drives and the University of Wyoming
GPS logs) and `2026/03_instrument_aligned/` (the LANL van), and writes only to a
temporary directory. It takes a few minutes, most of it reading ten days of
1 Hz files.

**It is committed without outputs**, because its outputs are the archive's data.
Run it to see them.

In [ ]:
import logging
import os
import tempfile
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from tsara import setup_logging
from tsara.align import (
    TsaraAlignError,
    attach_positions,
    bin_streams_onto_cells,
    build_output_grid,
    grid_cells,
    interpolate_onto_cells,
    load_grid,
    pair_species,
    save_grid,
)
from tsara.config.analysis import OutputGridConfig
from tsara.config.loader import load_manifest
from tsara.core.naming import sigma_sys_name
from tsara.core.support import CellBounds, overlap_pairs
from tsara.ingest import ingest_campaign

ARCHIVE_ENV = "TSARA_ARCHIVE"
if not os.environ.get(ARCHIVE_ENV):
    raise RuntimeError(
        f"This notebook reads the campaign archive, and {ARCHIVE_ENV} is not set. Set it to "
        "the directory holding the archive's 2024/ and 2026/ trees, e.g. "
        f"`export {ARCHIVE_ENV}=/path/to/Data`, start Jupyter from that shell, and run again. "
        "Notebook 04 runs the same operations on generated data and needs nothing."
    )
ARCHIVE = Path(os.environ[ARCHIVE_ENV])

# The only trees this notebook reads.
DRIVES = ARCHIVE / "2024" / "NOAA_MobileLab_Drives"
WYOMING_GPS = (
    ARCHIVE
    / "2024"
    / "SLC-SOS"
    / "2024_mobile"
    / "Univ_Wyoming"
    / "Mobile_Lab_State_Variables"
    / "ICARTT_GPS"
)
VAN_2026 = ARCHIVE / "2026" / "03_instrument_aligned"
missing = [
    str(tree.relative_to(ARCHIVE)) for tree in (DRIVES, WYOMING_GPS, VAN_2026) if not tree.is_dir()
]
if missing:
    raise RuntimeError(
        f"{ARCHIVE_ENV} is set, but it holds no {missing}. Point it at the "
        "directory containing 2024/ and 2026/."
    )

_work = tempfile.TemporaryDirectory()
WORK = Path(_work.name)
SECOND = 1_000_000_000

# --- figure style (identical to notebooks 01-04) ---------------------------
C1, C2, C3 = "#2a78d6", "#eb6834", "#1baf7a"
INK, INK2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, SURFACE = "#e1e0d9", "#fcfcfb"

plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "savefig.facecolor": SURFACE,
        "axes.edgecolor": "#c3c2b7",
        "axes.linewidth": 0.8,
        "axes.labelcolor": INK2,
        "axes.titlecolor": INK,
        "axes.titlesize": 11,
        "axes.titleweight": "normal",
        "axes.titlelocation": "left",
        "axes.labelsize": 9.5,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "xtick.labelsize": 8.5,
        "ytick.labelsize": 8.5,
        "grid.color": GRID,
        "grid.linewidth": 0.7,
        "grid.linestyle": "-",
        "legend.frameon": False,
        "legend.fontsize": 9,
        "font.size": 9.5,
        "lines.linewidth": 1.4,
        "figure.dpi": 110,
    }
)


def finish(
    ax, title=None, sub=None, ylab=None, xlab=None, legend=False, grid_axis="y", loc="upper right"
):
    """Apply the shared chrome: left-aligned title, muted subtitle, hairline grid."""
    if title:
        ax.set_title(title, pad=19 if sub else 8)
    if sub:
        ax.text(0, 1.025, sub, transform=ax.transAxes, fontsize=8.8, color=MUTED, va="bottom")
    if ylab:
        ax.set_ylabel(ylab)
    if xlab:
        ax.set_xlabel(xlab)
    if grid_axis:
        ax.grid(True, axis=grid_axis, alpha=0.9)
    ax.set_axisbelow(True)
    if legend:
        ax.legend(loc=loc)


def hhmm(ax, fmt="%H:%M"):
    """Label the x axis as clock time; the date belongs in the title."""
    ax.xaxis.set_major_formatter(mdates.DateFormatter(fmt))


def bounds_of(stream):
    """Cell start and stop of a stream, as datetime64 arrays for plotting."""
    edges = stream["time_bnds"].values
    return edges[:, 0], edges[:, 1]


def cells_of(stream):
    """Return a stream's cells as the CellBounds the binning primitive takes."""
    edges = stream["time_bnds"].values.astype("datetime64[ns]").astype("int64")
    return CellBounds(start_ns=edges[:, 0].copy(), stop_ns=edges[:, 1].copy())


def break_gaps(clock, values, tolerance=5.0):
    """Insert a NaN wherever the clock jumps, so no line is drawn across a gap."""
    clock, values = np.asarray(clock), np.asarray(values, dtype=float)
    steps = np.diff(clock).astype("timedelta64[ms]").astype(float)
    gaps = np.where(steps > tolerance * np.median(steps))[0]
    if gaps.size == 0:
        return clock, values
    return np.insert(clock, gaps + 1, clock[gaps]), np.insert(values, gaps + 1, np.nan)


def manifest(name, body):
    """Write a manifest into the scratch directory and load it."""
    path = WORK / f"{name}.yaml"
    path.write_text(body)
    return load_manifest(path)


def hide_paths(record):
    """Show the archive and scratch directories as <archive> and <work> in log records.

    On the handler rather than the logger: a record from `tsara.ingest` never
    consults the filters of the `tsara` logger it propagates to.
    """
    for real, shown in ((str(WORK), "<work>"), (str(ARCHIVE), "<archive>")):
        if isinstance(record.msg, str):
            record.msg = record.msg.replace(real, shown)
        if isinstance(record.args, tuple):
            record.args = tuple(
                str(a).replace(real, shown) if real in str(a) else a for a in record.args
            )
    return True


for handler in setup_logging(logging.WARNING).handlers:
    handler.addFilter(hide_paths)

# Every archive number METHODS §11 quotes is collected here as it is re-measured,
# and listed at the end beside the documented value.
LEDGER = []


def check(section, claim, documented, measured, fmt="{}"):
    """Record a documented number beside its re-measurement, compared as printed."""
    shown_doc, shown_now = fmt.format(documented), fmt.format(measured)
    LEDGER.append((section, claim, shown_doc, shown_now, shown_doc == shown_now))
    print(
        f"  [{'agrees' if shown_doc == shown_now else 'DIFFERS':>7}] {claim}: {shown_now} "
        f"(METHODS {section}: {shown_doc})"
    )


print(
    "archive found; reading",
    ", ".join(str(t.relative_to(ARCHIVE)) for t in (DRIVES, WYOMING_GPS, VAN_2026)),
)

---
## 1. One drive, ingested the way a manifest describes it

The 2024-07-18 drive of the NOAA mobile laboratory, with the instruments this
notebook needs. Every file on a drive day shares one 1 s merge grid, with a
blank wherever an instrument had nothing to report, so the files look alike and
the instruments do not.

Two declarations below are worth reading before the output.

* **`CH4_i_ppb` is loaded as `role: aux`.** The Picarro file carries its
  methane twice: `CH4_ppb` as measured, and `CH4_i_ppb` interpolated into every
  row. TSARA cannot tell them apart, so a manifest naming the `_i` column as a
  gas would feed it exactly the interpolated gas it exists never to produce
  (METHODS §9.2.3). It is loaded here only to be compared.
* **NOy's per-point `NOy_LIF_1SigmaAccuracy` is declared systematic.** It is
  the archive's only kind of per-point uncertainty column, and the file's own
  header says it combines the calibration uncertainty (±10 % for NOy) with the
  zero uncertainty (100 ppt): both are errors shared by neighbouring readings,
  which do not average down. The file writes NOy in pptv, so the manifest
  converts it, and the reported column follows the same scale.

In [ ]:
DAY = DRIVES / "20240718"
drive = ingest_campaign(
    manifest(
        "drive_20240718",
        f"""
name: drive_20240718
base_path: {DAY}
platform: {{kind: mobile, gps_instrument: metnav}}
instruments:
  picarro:
    loader: {{format: icartt, path_template: "USOS-Picarro-CO2-CH4-CO-H2O_MobileLab_*.ict"}}
    variables:
      co2: {{column: CO2_ppm, role: gas, units: ppm}}
      ch4: {{column: CH4_ppb, role: gas, units: ppb}}
      ch4_interpolated: {{column: CH4_i_ppb, role: aux, units: ppb}}
  lif:
    loader: {{format: icartt, path_template: "USOS-NOy-LIF_MobileLab_*.ict"}}
    variables:
      noy:
        column: NOy_LIF
        role: gas
        units: ppb
        convert: {{from_unit: pptv, to_unit: ppb, scale: 0.001}}
        uncertainty:
          systematic: {{mode: reported, column: NOy_LIF_1SigmaAccuracy}}
  ozone:
    loader: {{format: icartt, path_template: "USOS-O3_MobileLab_*.ict"}}
    variables:
      o3: {{column: O3_ppb, role: gas, units: ppb}}
  ptr:
    loader: {{format: icartt, path_template: "USOS-PTR_MobileLab_*.ict"}}
    variables:
      benzene_ptr: {{column: Benzene_NOAAPTR_ppbv, role: gas, units: ppb}}
  metnav:
    loader: {{format: icartt, path_template: "USOS-MetNav_MobileLab_*.ict"}}
    variables:
      latitude: {{column: GPS_Lat_deg, role: gps_lat, units: degrees_north}}
      longitude: {{column: GPS_Lon_deg, role: gps_lon, units: degrees_east}}
      wind_dir: {{column: WindDir_calc_deg, role: met, units: degrees, circular: true}}
      wind_speed: {{column: WindSpd_calc_m_s, role: met, units: m s-1}}
      air_temp: {{column: AirTemp_C, role: met, units: degC}}
  iwas:
    loader:
      format: icartt
      path_template: "USOS-iWAS_MobileLab_*.ict"
      support: {{stop_column: iWAS_Stop_UTC, method: mean}}
    variables:
      benzene_iwas: {{column: Benzene_ppbv, role: gas, units: ppb}}
      toluene_iwas: {{column: Toluene_ppbv, role: gas, units: ppb}}
""",
    )
)

print(
    f"\n{'stream':9} {'rows':>7} {'median cell':>12} {'label (source)':>22}  "
    "finite share per variable"
)
for name, stream in drive.items():
    widths = np.diff(stream["time_bnds"].values.astype("datetime64[ns]").astype("int64"), axis=1)
    label = f"{stream.attrs['tsara_support_label']} ({stream.attrs['tsara_support_label_source']})"
    shares = ", ".join(
        f"{v} {np.isfinite(stream[v].values).mean():.0%}"
        for v in stream.data_vars
        if not str(v).startswith("sigma_")
    )
    print(
        f"{name:9} {stream.sizes['time']:>7} {np.median(widths) / SECOND:>10.1f} s "
        f"{label:>22}  {shares}"
    )

In [ ]:
picarro = drive["picarro"]
check(
    "§9.2.3",
    "measured Picarro CH₄ rows holding a value",
    0.43,
    float(np.isfinite(picarro["ch4"].values).mean()),
    "{:.0%}",
)
check(
    "§9.2.3",
    "interpolated CH4_i_ppb rows holding a value",
    1.0,
    float(np.isfinite(picarro["ch4_interpolated"].values).mean()),
    "{:.0%}",
)
check(
    "§9.2.3",
    "measured O₃ rows holding a value",
    0.50,
    float(np.isfinite(drive["ozone"]["o3"].values).mean()),
    "{:.0%}",
)

both_there = np.isfinite(picarro["ch4"].values)
copy_gap = np.abs(picarro["ch4_interpolated"].values - picarro["ch4"].values)[both_there]
print(
    f"\nwhere a measured reading exists, the _i copy differs from it by a median "
    f"{np.median(copy_gap):.2f} ppb and at most {copy_gap.max():.1f} ppb"
)

# Two minutes around the drive's largest measured methane reading.
peak = pd.Timestamp(picarro["time"].values[int(np.nanargmax(picarro["ch4"].values))])
view = picarro.sel(time=slice(peak - pd.Timedelta("60s"), peak + pd.Timedelta("60s")))
measured = np.isfinite(view["ch4"].values)

fig, ax = plt.subplots(figsize=(9.5, 3.8))
ax.plot(
    view["time"].values,
    view["ch4_interpolated"].values,
    color=MUTED,
    lw=1.0,
    label="CH4_i_ppb: a value in every row, interpolated",
)
ax.plot(
    view["time"].values[measured],
    view["ch4"].values[measured],
    "o",
    color=C2,
    ms=4,
    label="CH4_ppb: what the analyzer reported",
)
finish(
    ax,
    "The same methane, measured and interpolated",
    "2024-07-18, two minutes around the drive's largest reading",
    "CH₄ (ppb)",
    "time (UTC)",
    legend=True,
    loc="upper left",
)
hhmm(ax, "%H:%M:%S")
plt.show()

Every stream arrives with cells: one second wide for the merge-grid
instruments, the exact fill intervals for the canisters, whose file names each
fill's stop time. The NOy-LIF file writes `Time_Mid` and the others
`Time_Start`, so TSARA's LIF cells sit half a second from everyone else's; that
matters in section 6.

The figure is the trap in one picture. Between the orange readings the grey
line is a ramp nobody measured, and a manifest naming `CH4_i_ppb` as a gas
would hand TSARA every point on it. Nor is the copy simply the readings joined
up: where a reading exists, the printed difference says how far the copy sits
from it, and at the peaks it does not pass through them.

---
## 2. A minute of methane, and what coverage says about it

Mirrors notebook 04 §2: the 1 Hz record averaged onto minute cells, with
`n_source` and `coverage` beside it. Here the cells come from `grid_cells`, the
epoch-anchored minutes a 60 s grid would use, and the interpolated copy is
binned beside the measured column for contrast.

In [ ]:
minutes = grid_cells(drive, OutputGridConfig(freq="60s"), ["ch4"])
on_minutes = bin_streams_onto_cells(drive, minutes, ["ch4", "ch4_interpolated"])
held = on_minutes["n_source_ch4"].values > 0

print(f"{len(minutes)} minute cells")
for column in ("ch4", "ch4_interpolated"):
    n = on_minutes[f"n_source_{column}"].values[held]
    cov = on_minutes[f"coverage_{column}"].values[held]
    print(
        f"  {column:17} median n_source {np.median(n):4.0f}, median coverage {np.median(cov):.2f}"
    )
check(
    "§11.7",
    "median readings per minute of measured CH₄",
    26,
    float(np.median(on_minutes["n_source_ch4"].values)),
    "{:.0f}",
)

rise = pd.Timestamp(on_minutes["time"].values[int(np.nanargmax(on_minutes["ch4"].values))])
window = (rise.floor("min") - pd.Timedelta("4min"), rise.floor("min") + pd.Timedelta("5min"))
cells_win = on_minutes.sel(time=slice(*window))
c0, c1 = bounds_of(cells_win)
raw = picarro.sel(time=slice(*window))
finite = np.isfinite(raw["ch4"].values)

fig, (ax, axc) = plt.subplots(
    2, 1, figsize=(9.5, 5.4), sharex=True, gridspec_kw={"height_ratios": [2.3, 1.0], "hspace": 0.3}
)
ax.hlines(cells_win["ch4"].values, c0, c1, color=C2, lw=3.4, zorder=2, label="binned onto minutes")
ax.plot(
    raw["time"].values[finite],
    raw["ch4"].values[finite],
    ".",
    color=C1,
    ms=3,
    zorder=3,
    label="measured readings",
)
finish(
    ax,
    "Real 1 Hz methane on minute cells",
    "2024-07-18; the analyzer reports every second or third row",
    "CH₄ (ppb)",
    legend=True,
    loc="upper left",
)
middle = c0 + (c1 - c0) / 2
width = (c1 - c0) * 0.4
axc.bar(
    middle - width / 2,
    cells_win["coverage_ch4"].values,
    width=width,
    color=C1,
    alpha=0.5,
    label="measured column",
)
axc.bar(
    middle + width / 2,
    cells_win["coverage_ch4_interpolated"].values,
    width=width,
    color=MUTED,
    alpha=0.5,
    label="interpolated column",
)
axc.set_ylim(0, 1.5)
finish(axc, None, None, "coverage", "time (UTC)", legend=True, loc="upper left")
axc.legend(loc="upper left", ncols=2)
hhmm(axc)
plt.show()

The measured column reports what the analyzer did: a value in roughly two
seconds of every five, so a minute rests on about 26 readings covering well
under half of it. The interpolated column reports 60 readings and full coverage
for every minute. Neither figure is computed wrongly; the second one is simply
describing interpolated rows, which is why the manifest has to choose the
column.

A reading every 2–3 s inside 1 s cells also means TSARA's inferred cell width
for the Picarro, one second, describes the merge grid rather than the analyzer's
own averaging. That is a width-inference question from Phase 3.5, noted rather
than changed here.

---
## 3. The direction that is refused

Mirrors notebook 04 §4. A canister fill is a 15 s mean. Put on the Picarro's
1 s cells, each fill would fill fifteen rows, each claiming full coverage.

In [ ]:
try:
    bin_streams_onto_cells(drive, cells_of(picarro), ["benzene_iwas"])
except TsaraAlignError as refused:
    print(refused)

---
## 4. An uncertainty the file reports

Mirrors notebook 04 §5, with the one per-point uncertainty column the archive
has. Declared systematic, it is carried through the minute average as a
weighted mean of the per-reading figures rather than reduced by √N.

In [ ]:
lif = drive["lif"]
noy_minutes = bin_streams_onto_cells(drive, minutes, ["noy"])
sigma = noy_minutes[sigma_sys_name("noy")]
print("attributes of the binned systematic sigma:")
for key in (
    "uncertainty_component",
    "uncertainty_source",
    "tsara_sigma_at_support",
    "tsara_propagation_form",
):
    print(f"    {key:24} {sigma.attrs.get(key)}")

per_reading = lif[sigma_sys_name("noy")]
print(f"\nper reading : median {float(per_reading.median()):.3f} ppb")
print(
    f"per minute  : median {float(sigma.median()):.3f} ppb, over a median of "
    f"{float(np.median(noy_minutes['n_source_noy'].values)):.0f} readings"
)
as_random = sigma.values / np.sqrt(noy_minutes["n_source_noy"].values)
print(
    f"the same column treated as random and independent: median {np.nanmedian(as_random):.3f} "
    f"ppb per minute, {float(sigma.median()) / np.nanmedian(as_random):.1f} times smaller"
)

The minute's systematic sigma stays at the level of its readings, labelled
`reported`; it is a weighted mean of them, so a minute holding a larger reading
carries a larger figure. Treating the same column as random would have claimed a
minute about eight times better determined. Which is right is a property of the
instrument rather than of TSARA, which is why the manifest has to say.

---
## 5. Pairing a canister against the analyzer

Mirrors notebook 04 §8. Benzene from the canisters against the analyzer's
measured methane. METHODS §11.4 quotes this pairing and checks it against a
loop written from the definition; both are re-run here.

In [ ]:
canister = pair_species(drive, "benzene_iwas", "ch4")
pairs = canister.dataset
print(
    f"clock {canister.clock} ({pairs.attrs['tsara_pairing_clock_reason']}), "
    f"{canister.n_pairs} pairs\n"
)
check("§11.4", "canister pairs on 2024-07-18", 32, canister.n_pairs)
check(
    "§11.4",
    "median coverage of a fill by measured CH₄",
    0.43,
    float(np.median(pairs["coverage_ch4"].values)),
    "{:.2f}",
)
check(
    "§11.4", "least coverage of a fill", 0.395, float(pairs["coverage_ch4"].values.min()), "{:.3f}"
)
check(
    "§11.4",
    "median CH₄ readings per fill",
    7,
    float(np.median(pairs["n_source_ch4"].values)),
    "{:.0f}",
)

# The same pairs from the definition: every 1 s cell weighted by its overlap with
# the fill, masked readings contributing nothing.
fills = cells_of(pairs)
cell_start, cell_stop = cells_of(picarro).start_ns, cells_of(picarro).stop_ns
methane = picarro["ch4"].values
by_loop, shortcut = [], []
for start, stop in zip(fills.start_ns, fills.stop_ns):
    overlap = np.clip(np.minimum(cell_stop, stop) - np.maximum(cell_start, start), 0, None)
    use = (overlap > 0) & np.isfinite(methane)
    by_loop.append((overlap[use] * methane[use]).sum() / overlap[use].sum())
    # The obvious shortcut: an unweighted mean of readings starting inside the fill.
    inside = (cell_start >= start) & (cell_start < stop) & np.isfinite(methane)
    shortcut.append(methane[inside].mean())
by_loop, shortcut = np.array(by_loop), np.array(shortcut)
check(
    "§11.4",
    "largest difference, TSARA against the loop (ppb)",
    0.0,
    float(np.max(np.abs(pairs["ch4"].values - by_loop))),
    "{:.3e}",
)
check(
    "§11.4",
    "largest difference, loop against the shortcut (ppb)",
    38.08,
    float(np.max(np.abs(by_loop - shortcut))),
    "{:.2f}",
)

try:
    pair_species(drive, "benzene_iwas", "ch4", min_coverage=0.9)
except TsaraAlignError as refused:
    print(f"\nmin_coverage=0.9: {refused}")

In [ ]:
k = int(np.nanargmax(pairs["benzene_iwas"].values))
centre = pd.Timestamp(pairs["time"].values[k])
span = (centre - pd.Timedelta("150s"), centre + pd.Timedelta("150s"))
trace = picarro.sel(time=slice(*span))
ok = np.isfinite(trace["ch4"].values)
near = pairs.sel(time=slice(*span))
f0, f1 = bounds_of(near)

fig = plt.figure(figsize=(10.8, 4.8))
gs = fig.add_gridspec(
    2, 2, width_ratios=[2.1, 1.0], height_ratios=[1.6, 1.0], hspace=0.35, wspace=0.28
)
ax = fig.add_subplot(gs[0, 0])
axb = fig.add_subplot(gs[1, 0], sharex=ax)
axs = fig.add_subplot(gs[:, 1])
ax.plot(
    trace["time"].values[ok],
    trace["ch4"].values[ok],
    "o",
    color=C1,
    ms=3.2,
    label="measured CH₄ readings",
)
for x0, x1 in zip(f0, f1):
    ax.axvspan(x0, x1, color=C3, alpha=0.2, lw=0)
    axb.axvspan(x0, x1, color=C3, alpha=0.2, lw=0)
ax.hlines(near["ch4"].values, f0, f1, color=C2, lw=4.0, label="CH₄ averaged over each fill")
finish(
    ax,
    "Real canister fills on the analyzer's record",
    "2024-07-18, five minutes around the drive's largest benzene fill",
    "CH₄ (ppb)",
    legend=True,
    loc="upper left",
)
axb.hlines(near["benzene_iwas"].values, f0, f1, color=C3, lw=4.0)
axb.set_ylim(0, float(np.nanmax(near["benzene_iwas"].values)) * 1.3)
finish(axb, None, None, "benzene (ppb)", "time (UTC)")
hhmm(axb, "%H:%M:%S")
points = axs.scatter(
    pairs["ch4"].values,
    pairs["benzene_iwas"].values,
    c=pairs["coverage_ch4"].values,
    cmap="viridis",
    s=26,
)
fig.colorbar(points, ax=axs, label="coverage of the fill by CH₄ readings")
finish(
    axs,
    f"All {canister.n_pairs} fills",
    "one pair per fill",
    "benzene (ppb)",
    "CH₄ (ppb)",
    grid_axis="both",
)
plt.show()

TSARA's pairs and the loop agree exactly, and the shortcut does not: giving an
edge reading full weight or none moves a fill's methane by tens of ppb, against
enhancements that are the whole point of a canister. Coverage says what the
figure shows, a fill sampled by half a dozen readings. And the reason
`min_coverage` defaults to zero is the refusal printed above: on this record a
threshold of 0.9 leaves no canister data at all.

---
## 6. Two 1 s clocks half a second apart

Mirrors notebook 04 §9, on the drive that prompted it. NOy-LIF is dense and
mid-labelled; the Picarro's CO₂ is sparse and start-labelled. The code as first
written gave a different number of pairs depending on which species was named
first. The notebook-04 section explains why that matters to a fit's
uncertainty; this one shows it on the files.

In [ ]:
for y, x in (("noy", "co2"), ("co2", "noy"), ("noy", "o3"), ("o3", "noy")):
    p = pair_species(drive, y, x)
    readings = {p.y_name: p.y_readings, p.x_name: p.x_readings}
    print(
        f"pair_species(drive, {y!r:6}, {x!r:6}) -> clock {p.clock:8} {p.n_pairs:>6} pairs; "
        f"readings {readings}"
    )

# What naming NOy first used to do: the same join on the LIF's cells.
for sparse in ("co2", "o3"):
    old = bin_streams_onto_cells(drive, cells_of(lif), [sparse, "noy"])
    both = np.isfinite(old[sparse].values) & np.isfinite(old["noy"].values)
    instrument = "picarro" if sparse == "co2" else "ozone"
    link = overlap_pairs(cells_of(drive[instrument]), cells_of(old.isel(time=np.flatnonzero(both))))
    real = np.isfinite(drive[instrument][sparse].values[link.source_index]) & (link.overlap_ns > 0)
    distinct = np.unique(link.source_index[real]).size
    print(
        f"\non the LIF clock, the old tie-break: {sparse} {int(both.sum())} pairs "
        f"from {distinct} distinct readings"
    )
    if sparse == "co2":
        check("§11.4.1", "NOy/CO₂ pairs on the LIF clock", 16893, int(both.sum()))
        check("§11.4.1", "distinct CO₂ readings behind them", 8447, distinct)
    else:
        check("§11.4.1", "NOy/O₃ pairs on the LIF clock", 19498, int(both.sum()))
check(
    "§11.4.1", "NOy/CO₂ pairs on the rule's clock", 8447, pair_species(drive, "noy", "co2").n_pairs
)
check("§11.4.1", "NOy/O₃ pairs on the rule's clock", 9750, pair_species(drive, "noy", "o3").n_pairs)

Both orders now give the sparser member's clock, and every CO₂ reading is one
pair. On the LIF's clock the same readings made twice as many pairs, each
counted in two of them. Nothing in either run looks wrong on its own, which is
the reason the clock is now chosen by rule and the readings are recorded.

---
## 7. Ten drives

The rest of the notebook needs all ten 2024 drive days: the Picarro, the
MetNav record (position, ground speed, wind) and the canisters. One manifest
covers them, with `*` standing for the day directory.

In [ ]:
ten = ingest_campaign(
    manifest(
        "ten_drives",
        f"""
name: ten_drives
base_path: {DRIVES}
platform: {{kind: mobile, gps_instrument: metnav}}
instruments:
  picarro:
    loader: {{format: icartt, path_template: "*/USOS-Picarro-CO2-CH4-CO-H2O_MobileLab_*.ict"}}
    variables:
      co2: {{column: CO2_ppm, role: gas, units: ppm}}
      ch4: {{column: CH4_ppb, role: gas, units: ppb}}
  metnav:
    loader: {{format: icartt, path_template: "*/USOS-MetNav_MobileLab_*.ict"}}
    variables:
      latitude: {{column: GPS_Lat_deg, role: gps_lat, units: degrees_north}}
      longitude: {{column: GPS_Lon_deg, role: gps_lon, units: degrees_east}}
      ground_speed: {{column: GPS_GndSpd_m_s, role: aux, units: m s-1}}
      wind_dir: {{column: WindDir_calc_deg, role: met, units: degrees, circular: true}}
      wind_speed: {{column: WindSpd_calc_m_s, role: met, units: m s-1}}
  iwas:
    loader:
      format: icartt
      path_template: "*/USOS-iWAS_MobileLab_*.ict"
      support: {{stop_column: iWAS_Stop_UTC, method: mean}}
    variables:
      benzene_iwas: {{column: Benzene_ppbv, role: gas, units: ppb}}
""",
    )
)
met = ten["metnav"]
starts = cells_of(met).start_ns
# Drives cross midnight UTC, so a drive is a run of rows without an hour's gap,
# not a calendar date.
drive_number = np.concatenate([[0], np.cumsum(np.diff(starts) > 3600 * SECOND)])
fills_all = cells_of(ten["iwas"])
print(
    {name: stream.sizes["time"] for name, stream in ten.items()},
    f"over {drive_number.max() + 1} drives",
)
check("§11.4", "canister fills on the ten drive days", 261, len(fills_all))
check(
    "§11.4", "median fill width (s)", 14.9, float(np.median(fills_all.width_ns)) / SECOND, "{:.1f}"
)
fill_drive = drive_number[np.clip(np.searchsorted(starts, fills_all.start_ns), 0, starts.size - 1)]
between_fills = np.concatenate(
    [np.diff(fills_all.start_ns[fill_drive == n]) for n in np.unique(fill_drive)]
)
check(
    "§11.4",
    "median time between fills within a drive (s)",
    530,
    float(np.median(between_fills)) / SECOND,
    "{:.0f}",
)

---
## 8. Wind direction on minutes

Mirrors notebook 04 §10. The MetNav's 1 Hz wind direction over ten drives,
vector-averaged onto minutes. The arithmetic mean it replaces is computed with
the same join and the same weights, on a copy of the variable that does not
declare itself circular.

In [ ]:
wind_cells = grid_cells(ten, OutputGridConfig(freq="60s"), ["wind_dir"])
arithmetic_copy = met.assign(
    wind_dir_arithmetic=("time", met["wind_dir"].values, {"role": "met", "units": "degrees"})
)
wind = bin_streams_onto_cells(
    {"metnav": arithmetic_copy}, wind_cells, ["wind_dir", "wind_dir_arithmetic"]
)
enough = wind["n_source_wind_dir"].values >= 30
apart = np.abs((wind["wind_dir_arithmetic"].values - wind["wind_dir"].values + 180) % 360 - 180)
apart, R = apart[enough], wind["wind_dir_resultant_length"].values[enough]
check("§11.5", "minutes with at least 30 readings", 3337, int(enough.sum()))
check(
    "§11.5",
    "arithmetic mean more than 45° from the vector mean",
    0.267,
    float(np.mean(apart > 45)),
    "{:.1%}",
)
check("§11.5", "median disagreement (degrees)", 7.0, float(np.median(apart)), "{:.1f}")

dispersion = wind["wind_dir_dispersion"].values[enough]
classes = (
    ("R > 0.99", R > 0.99, 58, 7.1),
    ("R 0.90-0.99", (R > 0.9) & (R <= 0.99), 1140, 18.5),
    ("R 0.50-0.90", (R >= 0.5) & (R <= 0.9), 1713, 40.6),
    ("R < 0.50", R < 0.5, 426, 80.5),
)
for label, members, cells_doc, sd_doc in classes:
    check("§11.5", f"minutes with {label}", cells_doc, int(members.sum()))
    check(
        "§11.5",
        f"median dispersion, {label} (degrees)",
        sd_doc,
        float(np.median(dispersion[members])),
        "{:.1f}",
    )

In [ ]:
# The 2024-07-18 drive only: unit vectors against speed-weighted vectors, the
# second built by binning the wind components as ordinary scalars.
first = met.isel(time=np.flatnonzero(drive_number == 0))
speed, direction = first["wind_speed"].values, first["wind_dir"].values
components = first.assign(
    u=("time", speed * np.sin(np.radians(direction)), {"role": "met"}),
    v=("time", speed * np.cos(np.radians(direction)), {"role": "met"}),
)
first_minutes = grid_cells({"metnav": first}, OutputGridConfig(freq="60s"), ["wind_dir"])
both_ways = bin_streams_onto_cells({"metnav": components}, first_minutes, ["wind_dir", "u", "v"])
weighted = np.degrees(np.arctan2(both_ways["u"].values, both_ways["v"].values)) % 360
usable = (both_ways["n_source_wind_dir"].values >= 30) & np.isfinite(weighted)
gap = np.abs((weighted - both_ways["wind_dir"].values + 180) % 360 - 180)[usable]
check("§11.5", "07-18 minutes compared, unit against speed-weighted", 325, int(usable.sum()))
check("§11.5", "median difference (degrees)", 2.2, float(np.median(gap)), "{:.1f}")
check("§11.5", "largest difference (degrees)", 54.6, float(gap.max()), "{:.1f}")

hour_start = pd.Timestamp(first["time"].values[0]).ceil("h")
hour = wind.sel(time=slice(hour_start, hour_start + pd.Timedelta("1h")))
h0, h1 = bounds_of(hour)
raw_hour = first.sel(time=slice(hour_start, hour_start + pd.Timedelta("1h")))

fig, (ax, axr) = plt.subplots(
    2, 1, figsize=(10.0, 5.6), sharex=True, gridspec_kw={"height_ratios": [2.4, 1.0], "hspace": 0.3}
)
ax.plot(
    raw_hour["time"].values,
    raw_hour["wind_dir"].values,
    ".",
    color=C1,
    ms=1.4,
    alpha=0.3,
    label="1 Hz wind direction",
)
ax.hlines(
    hour["wind_dir_arithmetic"].values,
    h0,
    h1,
    color=C3,
    lw=2.2,
    ls=(0, (3, 1.5)),
    label="arithmetic mean of each minute",
)
ax.hlines(hour["wind_dir"].values, h0, h1, color=C2, lw=3.2, label="vector mean (TSARA)")
# Headroom above 360 for the legend, so it covers no data.
ax.set_ylim(0, 430)
ax.set_yticks([0, 90, 180, 270, 360])
finish(
    ax,
    "Real wind direction from a moving van",
    "the first full hour of the 2024-07-18 drive",
    "direction (degrees)",
)
ax.legend(loc="upper left", ncols=3, fontsize=8.2, frameon=True, facecolor=SURFACE, edgecolor=GRID)
axr.plot(hour["time"].values, hour["wind_dir_resultant_length"].values, "o-", color=C2, ms=3)
axr.set_ylim(0, 1.05)
finish(axr, None, None, "resultant length R", "time (UTC)")
hhmm(axr)
plt.show()

On a moving platform a minute's wind direction is usually not a well-determined
number, and the lower panel says so minute by minute. The printed classes
count the same thing over ten drives. Part of that spread is the van turning
rather than the air, which TSARA does not try to separate (METHODS §11.5).

---
## 9. Positions: attaching the track, and what the guard costs

Mirrors notebook 04 §11. Ingestion left the mobile platform's track on the
MetNav clock; `attach_positions` puts it on the canister fills under the gap
guard.

In [ ]:
iwas = drive["iwas"]
placed = attach_positions(iwas, drive, max_interp_gap="10s")
latitude = interpolate_onto_cells(
    drive, ("metnav", "latitude"), cells_of(iwas), max_interp_gap="10s"
)
print("coordinates now on the canister stream:", sorted(map(str, placed.coords)))
check("§11.6", "fills positioned", 32, int(np.isfinite(placed["latitude"].values).sum()))
check("§11.6", "fills landing exactly on a GPS second", 2, latitude.n_exact)
check("§11.6", "fills interpolated between two", 30, latitude.n_interpolated)

# How far the van moves during a fill: the summed 1 s path of the fixes inside it.
day_met = drive["metnav"]
fix_start = cells_of(day_met).start_ns
lat, lon = day_met["latitude"].values, day_met["longitude"].values
METRES = 111_320.0
path = []
for start, stop in zip(cells_of(iwas).start_ns, cells_of(iwas).stop_ns):
    inside = (fix_start >= start) & (fix_start <= stop) & np.isfinite(lat)
    steps = np.hypot(
        np.diff(lat[inside]) * METRES, np.diff(lon[inside]) * METRES * np.cos(np.radians(40.76))
    )
    path.append(steps.sum())
check("§11.6", "median path covered during a fill (m)", 127, float(np.median(path)), "{:.0f}")
check("§11.6", "longest path covered during a fill (m)", 307, float(np.max(path)), "{:.0f}")

The thinning table in METHODS §11.6 is the cost of the guard: every drive's
track, scored only while the van moves faster than 2 m/s, thinned to every
*k*-th finite fix, interpolated linearly, and compared with the removed fixes
lying strictly inside a thinned bracket exactly *k* seconds long.

In [ ]:
lat_all, lon_all = met["latitude"].values, met["longitude"].values
speed_all = met["ground_speed"].values
seconds_all = starts // SECOND
documented = {
    5: (117253, 2.8),
    10: (131146, 9.5),
    30: (139222, 54.7),
    50: (140183, 108),
    60: (140335, 137),
}
print(f"{'one fix every':>13} {'scored':>8} {'median':>8} {'p90':>8} {'max':>7}")
for k, (scored_doc, p90_doc) in documented.items():
    errors = []
    for number in range(drive_number.max() + 1):
        rows = np.flatnonzero(
            (drive_number == number) & np.isfinite(lat_all) & np.isfinite(lon_all)
        )
        kept = rows[::k]
        kept_t = seconds_all[kept]
        right = np.searchsorted(kept_t, seconds_all[rows], side="right")
        between = (right > 0) & (right < kept.size) & ~np.isin(rows, kept)
        left_fix, right_fix = kept[right[between] - 1], kept[right[between]]
        exact = (seconds_all[right_fix] - seconds_all[left_fix]) == k
        target, left_fix, right_fix = rows[between][exact], left_fix[exact], right_fix[exact]
        share = (seconds_all[target] - seconds_all[left_fix]) / k
        guess_lat = lat_all[left_fix] + share * (lat_all[right_fix] - lat_all[left_fix])
        guess_lon = lon_all[left_fix] + share * (lon_all[right_fix] - lon_all[left_fix])
        error = np.hypot(
            (guess_lat - lat_all[target]) * METRES,
            (guess_lon - lon_all[target]) * METRES * np.cos(np.radians(lat_all[target])),
        )
        errors.append(error[speed_all[target] > 2.0])
    error = np.concatenate(errors)
    print(
        f"{k:>11} s {error.size:>8} {np.median(error):>6.1f} m {np.percentile(error, 90):>6.1f} m "
        f"{error.max():>5.0f} m"
    )
    check("§11.6", f"positions scored at one fix every {k} s", scored_doc, error.size)
    check(
        "§11.6",
        f"90th percentile error at {k} s (m)",
        p90_doc,
        float(np.percentile(error, 90)),
        "{:.1f}" if k <= 30 else "{:.0f}",
    )

And the case where the guard can never be met: the University of Wyoming's
mobile laboratory logged GPS as a separate file per drive, several of them a fix
every 50 s. Against the default 10 s guard, TSARA warns.

In [ ]:
wyoming = sorted(WYOMING_GPS.glob("UWMobileLab_GPSdata_*.ict"))
spacing, positioned = {}, {}
auxiliary_log = logging.getLogger("tsara.align.auxiliary")
ingest_log = logging.getLogger("tsara.ingest")
# Some of these logs repeat a timestamp, which ingestion reports file by file;
# it is not what this cell is about.
ingest_log.setLevel(logging.ERROR)
for path in wyoming:
    gps = ingest_campaign(
        manifest(
            "wyoming_gps",
            f"""
name: wyoming_gps
base_path: {WYOMING_GPS}
platform: {{kind: stationary, latitude: 40.76, longitude: -111.89}}
instruments:
  gps:
    loader: {{format: icartt, path_template: "{path.name}"}}
    variables:
      gps_lat: {{column: LATITUDE, role: gps_lat, units: degrees_north}}
""",
        )
    )["gps"]
    fix_ns = cells_of(gps).start_ns[np.isfinite(gps["gps_lat"].values)]
    spacing[path.name] = float(np.median(np.diff(fix_ns))) / SECOND
    whole = np.arange(fix_ns[0] // SECOND * SECOND, fix_ns[-1], SECOND)
    seconds = CellBounds(start_ns=whole, stop_ns=whole + SECOND)
    # One file's warning is shown; the other twelve would repeat it.
    auxiliary_log.setLevel(logging.NOTSET if "_D06_" in path.name else logging.ERROR)
    field = interpolate_onto_cells({"gps": gps}, "gps_lat", seconds, max_interp_gap="10s")
    positioned[path.name] = float(np.isfinite(field.values).mean())
auxiliary_log.setLevel(logging.NOTSET)
ingest_log.setLevel(logging.NOTSET)

for name in spacing:
    print(
        f"  {name:42} a fix every {spacing[name]:5.1f} s; 1 s cells positioned "
        f"{positioned[name]:6.1%}"
    )
fifty = [name for name in spacing if spacing[name] >= 50]
check("§11.6", "Wyoming GPS files logging every 50 s", 7, len(fifty))
check(
    "§11.6",
    "1 s cells those files position at the 10 s guard",
    0.0,
    max(positioned[name] for name in fifty),
    "{:.1%}",
)

The warning above is for one of those files; the table is all thirteen. A
record sampled every 50 s cannot meet a 10 s guard anywhere, so the join
positions nothing, and until walkthrough stage 5 it said so only at INFO level.

---
## 10. The drive's matrix

Mirrors notebook 04 §12, with the selection METHODS §11.7 states: the Picarro's
measured CO₂ and CH₄, MetNav wind direction and air temperature, NOy-LIF, PTR-MS
benzene, and the canisters' benzene and toluene.

In [ ]:
SELECTION = [
    "co2",
    "ch4",
    "wind_dir",
    "air_temp",
    "noy",
    "benzene_ptr",
    "benzene_iwas",
    "toluene_iwas",
]
WITHOUT_CANISTERS = [v for v in SELECTION if not v.endswith("_iwas")]
for freq, chosen, documented_rows in (
    ("5s", SELECTION, None),
    ("15s", SELECTION, 1302),
    ("1s", WITHOUT_CANISTERS, 19502),
):
    try:
        rows = len(grid_cells(drive, OutputGridConfig(freq=freq), chosen))
        print(f"{freq:>4} over {len(chosen)} variables: {rows} cells")
        check("§11.7", f"{freq} grid cells", documented_rows, rows)
    except TsaraAlignError as refused:
        print(f"{freq:>4} over {len(chosen)} variables: refused -- {str(refused).split('. ')[0]}.")

matrix = build_output_grid(drive, OutputGridConfig(freq="60s"), SELECTION)
documented = {
    "co2": (0.994, 26),
    "ch4": (0.994, 26),
    "wind_dir": (0.997, 60),
    "air_temp": (0.997, 60),
    "noy": (1.0, 61),
    "benzene_ptr": (0.972, None),
    "benzene_iwas": (0.113, 0),
    "toluene_iwas": (0.113, 0),
}
print(f"\n{'variable':13} {'rows filled':>12} {'median n_source':>16} {'readings':>9}")
for variable, (share_doc, n_doc) in documented.items():
    counts = matrix[f"n_source_{variable}"].values
    print(
        f"{variable:13} {np.mean(counts > 0):>12.1%} {np.median(counts):>16.0f} "
        f"{matrix[variable].attrs['tsara_grid_readings']:>9}"
    )
    check("§11.7", f"60 s rows holding {variable}", share_doc, float(np.mean(counts > 0)), "{:.1%}")
    if n_doc is not None:
        check("§11.7", f"median n_source of {variable}", n_doc, float(np.median(counts)), "{:.0f}")

The matrix is honest about each instrument. Met and the LIF fill almost every
minute from about sixty readings; the Picarro fills them from about 26; the
canisters appear in one row in nine, and the warning above names them because
some fills straddle a minute boundary. The other warning above came from the
one-second request in the table: the LIF's half-second offset does not matter to
a minute row, which is far wider than its cells, but its cells are exactly the
period of a one-second grid.

In [ ]:
one_second = build_output_grid(drive, OutputGridConfig(freq="1s"), ["co2", "noy"])
print(
    f"n_source of noy on the 1 s grid: "
    f"{dict(zip(*np.unique(one_second['n_source_noy'].values, return_counts=True)))}"
)

Every LIF value on that grid is the mean of two neighbouring readings. Starting
the grid on the LIF's own boundaries would do the same to the Picarro, and which
instrument a 1 s grid should respect is the user's decision, made with `start`.

Notebook 04 §12 found that a joined product joined again forgets its coverage.
Here is the drive's minute matrix put onto five-minute rows, beside the same
rows built from the stream:

In [ ]:
five_from_grid = build_output_grid({"matrix": matrix}, OutputGridConfig(freq="300s"), ["ch4"])
five_from_stream = build_output_grid(drive, OutputGridConfig(freq="300s"), ["ch4"])
occupied = five_from_stream["n_source_ch4"].values > 0
print(
    f"median coverage of a 5-minute row by measured CH₄: from the stream "
    f"{np.median(five_from_stream['coverage_ch4'].values[occupied]):.2f}, from the minute "
    f"matrix {np.median(five_from_grid['coverage_ch4'].values[occupied]):.2f}"
)
moved = np.abs(five_from_stream["ch4"].values - five_from_grid["ch4"].values)[occupied]
print(
    f"rows whose value moves: {int((moved > 1e-9).sum())} of {int(occupied.sum())}, "
    f"largest {np.nanmax(moved):.2f} ppb"
)

---
## 11. Ten drives on one grid

Mirrors notebook 04 §12 and §13. A uniform grid spans the weeks between drives,
and that is where its size and its compression come from. Then the measurement
behind the deleted median option (METHODS §11.7.1), under its stated rule.

In [ ]:
THREE = ["co2", "ch4", "wind_dir"]
for freq, rows_doc, share_doc in (("60s", 42443, 0.079), ("1s", 2546521, 0.079)):
    campaign_grid = build_output_grid(ten, OutputGridConfig(freq=freq), THREE)
    any_data = np.zeros(campaign_grid.sizes["time"], dtype=bool)
    for variable in THREE:
        any_data |= campaign_grid[f"n_source_{variable}"].values > 0
    span_days = (
        cells_of(campaign_grid).stop_ns[-1] - cells_of(campaign_grid).start_ns[0]
    ) / 86400e9
    print(
        f"{freq:>4}: {campaign_grid.sizes['time']:,} rows over {span_days:.1f} days, "
        f"{any_data.mean():.1%} holding any data, {campaign_grid.nbytes / 1e6:.0f} MB"
    )
    check("§11.7", f"{freq} ten-drive grid rows", rows_doc, campaign_grid.sizes["time"])
    if freq == "60s":
        check(
            "§11.7",
            "60 s ten-drive rows holding any data",
            share_doc,
            float(any_data.mean()),
            "{:.1%}",
        )
    else:
        check("§11.7", "1 s ten-drive rows empty", 0.921, float(1 - any_data.mean()), "{:.1%}")

for level, size_doc in ((None, 285.2), (4, 3.7)):
    folder = WORK / f"ten_drives_{level}"
    written = save_grid(campaign_grid, folder, compression=level)
    size = written.stat().st_size / 1e6
    print(f"compression={level!s:>4}: {size:6.1f} MB")
    check(
        "§11.7", f"1 s ten-drive grid on disk, compression={level} (MB)", size_doc, size, "{:.1f}"
    )
same = load_grid(WORK / "ten_drives_4")
print(
    "identical after reload:",
    all(
        np.array_equal(campaign_grid[v].values, same[v].values, equal_nan=True)
        for v in campaign_grid.data_vars
    ),
)
del campaign_grid, same

with_canisters = build_output_grid(ten, OutputGridConfig(freq="60s"), THREE + ["benzene_iwas"])
link = overlap_pairs(fills_all, cells_of(with_canisters))
rows_per_fill = np.bincount(link.source_index[link.overlap_ns > 0], minlength=len(fills_all))
check(
    "§11.7",
    "60 s rows holding canister benzene",
    320,
    int((with_canisters["n_source_benzene_iwas"].values > 0).sum()),
)
check("§11.7", "fills landing in two rows", 68, int((rows_per_fill == 2).sum()))

In [ ]:
# METHODS §11.7.1, under its stated rule: each reading's enhancement is its value
# minus a rolling 10-minute 5th percentile of the readings; readings are grouped
# into epoch-aligned minutes; a minute is enhanced when its mean enhancement
# exceeds 5 ppb; a negative median counts as zero.
picarro_all = ten["picarro"]
reading_time = pd.DatetimeIndex(cells_of(picarro_all).start_ns)
series = pd.Series(picarro_all["ch4"].values, index=reading_time)
enhanced = []
for _, one_drive in series.groupby(drive_number):
    baseline = one_drive.rolling("600s", min_periods=60).quantile(0.05)
    excess = (one_drive - baseline).dropna()
    by_minute = excess.groupby(excess.index.floor("60s"))
    table = pd.DataFrame(
        {
            "mean": by_minute.mean(),
            "median": by_minute.median(),
            "above": by_minute.apply(lambda x: float((x > 5).mean())),
        }
    )
    enhanced.append(table[table["mean"] > 5])
enhanced = pd.concat(enhanced)
kept = enhanced["median"].clip(lower=0)
thin = enhanced["above"] < 0.5
zeroed = enhanced["median"] <= 0
check("§11.7.1", "enhanced minutes", 2147, len(enhanced))
check(
    "§11.7.1",
    "enhancement mass a 60 s median discards",
    0.196,
    float(1 - kept.sum() / enhanced["mean"].sum()),
    "{:.1%}",
)
check("§11.7.1", "minutes with fewer than half their readings above 5 ppb", 334, int(thin.sum()))
check(
    "§11.7.1",
    "what the median loses over those minutes",
    0.788,
    float(1 - kept[thin].sum() / enhanced["mean"][thin].sum()),
    "{:.1%}",
)
check("§11.7.1", "minutes a median reduces to zero", 20, int(zeroed.sum()))
check("§11.7.1", "largest of them (ppb)", 42.8, float(enhanced["mean"][zeroed].max()), "{:.1f}")

---
## 12. The 2026 van: an analyzer that is not quite one second

Mirrors the "too fine" part of notebook 04 §12. Two LANL Aeris methane
analyzers on the 2026-01-19 drive, from the aligned parquet stage, with the
quarantine directories excluded.

In [ ]:
van = ingest_campaign(
    manifest(
        "van_20260119",
        f"""
name: van_20260119
base_path: {VAN_2026}
platform: {{kind: stationary, latitude: 40.76, longitude: -111.89}}
instruments:
  pico:
    loader:
      format: parquet
      path_template: "LANL_aerispico017/Eng/Pico100017_260119_*Eng.parquet"
      exclude: ["**/bad/**", "**/bad_timestamp/**"]
    variables:
      ch4_pico: {{column: CH4_ppm, role: gas, units: ppm}}
  ultra:
    loader:
      format: parquet
      path_template: "LANL_aerisultra321/Eng/Ultra100321_260119_*Eng.parquet"
      exclude: ["**/bad/**", "**/bad_timestamp/**"]
    variables:
      ch4_ultra: {{column: CH4_ppm, role: gas, units: ppm}}
""",
    )
)
for name, stream in van.items():
    print(
        f"{name}: {stream.sizes['time']} cells, median width "
        f"{np.median(cells_of(stream).width_ns) / SECOND:.3f} s"
    )
check(
    "§11.7",
    "Aeris pico cell width (s)",
    1.023,
    float(np.median(cells_of(van["pico"]).width_ns)) / SECOND,
    "{:.3f}",
)
van_grid = build_output_grid(van, OutputGridConfig(freq="1s"))
check(
    "§11.7",
    "1 s rows holding ch4_pico",
    17517,
    int((van_grid["n_source_ch4_pico"].values > 0).sum()),
)
check(
    "§11.7",
    "ch4_pico readings behind them",
    17069,
    van_grid["ch4_pico"].attrs["tsara_grid_readings"],
)

A one-second grid is built for an analyzer reporting 1.023 s cells, with a few
percent more rows than readings, and the readings warning says which analyzer.
The comparison of widths that walkthrough stage 6 replaced would have refused it.

---
## 13. The ledger

Every archive number this notebook re-measured, beside what METHODS says. Each
is compared at the precision METHODS prints it.

In [ ]:
ledger = pd.DataFrame(LEDGER, columns=["METHODS", "claim", "documented", "measured", "agrees"])
with pd.option_context("display.max_rows", None, "display.max_colwidth", 70, "display.width", 200):
    print(ledger.to_string(index=False))
print(f"\n{int(ledger['agrees'].sum())} of {len(ledger)} agree")